1. Watermark   → ¿hasta dónde procesé la última vez?
2. Leer bronze → solo filas con _ingested_at > watermark
3. Tipar       → nombres snake_case, tipos del contrato, _change_ts
4. Calidad     → válidas siguen; inválidas → ops.data_quality_issues
5. Deduplicar  → ROW_NUMBER: última versión por transaction_id
6. MERGE       → inserta nuevas, actualiza solo si la versión es más reciente
   + guardar el watermark nuevo (solo si todo salió bien)

In [0]:
# ── Imports ──────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from lib.config import Config
from lib.audit import audit_run

# ── Parámetros (los rellena el Job o ADF; a mano toman el valor por defecto) ──
dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("job_run_id", "", "Id de ejecución del orquestador")

cfg = Config(catalog=dbutils.widgets.get("catalog"))
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

# ── Tablas involucradas ──────────────────────────────────────────────────
SOURCE     = cfg.table("bronze", "transactions")       # origen
TARGET     = cfg.table("silver", "transactions")       # destino
DQ_TABLE   = cfg.table("ops", "data_quality_issues")   # registros rechazados
WATERMARKS = cfg.table("ops", "watermarks")            # control incremental

# ── Reglas de calidad: nombre → condición que indica ERROR ───────────────
# Centralizadas acá para que sean fáciles de leer, cambiar y auditar.
REGLAS = {
    "transaction_id_nulo":  "transaction_id IS NULL",
    "customer_id_nulo":     "customer_id IS NULL",
    "quantity_no_positiva": "quantity IS NULL OR quantity <= 0",
    "datos_rescatados":     "_rescued_data IS NOT NULL",
}

In [0]:
with audit_run(cfg, "03_silver_transactions", "silver", TARGET, JOB_RUN_ID) as run:

    # ── 1. Watermark: último _ingested_at de bronze procesado con éxito ──
    # Es el mismo patrón que la tabla de control de Fabric/ADF.
    # Si es la primera ejecución (NULL), se procesa todo bronze.
    wm = spark.sql(f"""
        SELECT MAX(watermark_ts) AS wm
        FROM {WATERMARKS}
        WHERE source_table = '{SOURCE}' AND target_table = '{TARGET}'
    """).first()["wm"]

    # ── 2. Leer solo lo nuevo de bronze ──────────────────────────────────
    bronze = spark.table(SOURCE)
    nuevos = bronze if wm is None else bronze.filter(F.col("_ingested_at") > F.lit(wm))

    run.rows_read = nuevos.count()
    print(f"Watermark anterior: {wm} | filas nuevas en bronze: {run.rows_read}")

    if run.rows_read == 0:
        run.rows_written = 0
        run.rows_rejected = 0
        print("Sin datos nuevos: nada que hacer")
    else:
        # El nuevo watermark sale de los DATOS leídos (no del reloj)
        nuevo_wm = nuevos.agg(F.max("_ingested_at")).first()[0]

        # ── 3. Tipar y renombrar según el contrato de silver ─────────────
        # Nombres en snake_case, tipos explícitos (DECIMAL para dinero) y
        # _change_ts = COALESCE(updated_at, created_at) → versión del registro.
        tipado = nuevos.select(
            F.col("transactionID").cast("bigint").alias("transaction_id"),
            F.col("customerID").cast("bigint").alias("customer_id"),
            F.col("franchiseID").cast("bigint").alias("franchise_id"),
            F.col("dateTime").cast("timestamp").alias("transaction_ts"),
            F.to_date("dateTime").alias("transaction_date"),
            F.trim("product").alias("product"),
            F.col("quantity").cast("int").alias("quantity"),
            F.col("unitPrice").cast("decimal(10,2)").alias("unit_price"),
            F.col("totalPrice").cast("decimal(12,2)").alias("total_price"),
            F.lower(F.trim("paymentMethod")).alias("payment_method"),
            F.col("discountPct").cast("decimal(5,2)").alias("discount_pct"),
            F.col("createdAt").cast("timestamp").alias("created_at"),
            F.col("updatedAt").cast("timestamp").alias("updated_at"),
            F.coalesce(F.col("updatedAt").cast("timestamp"),
                       F.col("createdAt").cast("timestamp")).alias("_change_ts"),
            F.col("_source_file"),
            F.col("_ingested_at").alias("_bronze_ingested_at"),
            F.col("_rescued_data"),
        )

        # ── 4. Calidad: cada fila recibe la lista de reglas que incumple ──
        # array_compact quita los NULL → lista vacía = fila válida.
        evaluado = tipado.withColumn(
            "_reglas_incumplidas",
            F.array_compact(F.array(*[F.when(F.expr(cond), F.lit(nombre))
                                      for nombre, cond in REGLAS.items()])))

        validos   = evaluado.filter(F.size("_reglas_incumplidas") == 0)
        invalidos = evaluado.filter(F.size("_reglas_incumplidas") > 0)

        # Inválidos → ops.data_quality_issues (una fila por regla incumplida,
        # con el registro completo en JSON para poder analizarlo o reprocesarlo)
        (invalidos
            .select(
                F.lit(run.run_id).alias("run_id"),
                F.lit("silver").alias("layer"),
                F.lit(TARGET).alias("target_table"),
                F.explode("_reglas_incumplidas").alias("rule_name"),
                F.col("transaction_id").cast("string").alias("record_key"),
                F.to_json(F.struct("*")).alias("record_payload"),
                F.current_timestamp().alias("detected_at"))
            .write.mode("append").saveAsTable(DQ_TABLE))

        run.rows_rejected = invalidos.count()

        # ── 5. Deduplicar: la última versión de cada venta ───────────────
        # ROW_NUMBER por transaction_id, ordenado por la versión del origen
        # (_change_ts) y, como desempate, por la llegada a bronze.
        w = Window.partitionBy("transaction_id").orderBy(
                F.col("_change_ts").desc(), F.col("_bronze_ingested_at").desc())

        ultima_version = (validos
            .drop("_reglas_incumplidas", "_rescued_data")
            .withColumn("_rn", F.row_number().over(w))
            .filter("_rn = 1")
            .drop("_rn")
            .withColumn("_run_id", F.lit(run.run_id))
            .withColumn("_silver_updated_at", F.current_timestamp()))

        # ── 6. MERGE (upsert) en silver ──────────────────────────────────
        # - Si la venta no existe → INSERT.
        # - Si existe y la que llega es MÁS RECIENTE → UPDATE.
        # - Si existe y la que llega es igual o más vieja → no hace nada.
        #   (Protege contra datos que llegan desordenados y hace la carga idempotente.)
        ultima_version.createOrReplaceTempView("silver_upsert")
        spark.sql(f"""
            MERGE INTO {TARGET} AS t
            USING silver_upsert AS s
              ON t.transaction_id = s.transaction_id
            WHEN MATCHED AND s._change_ts > t._change_ts THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        run.rows_written = spark.table(TARGET).filter(F.col("_run_id") == run.run_id).count()

        # ── Guardar el watermark nuevo: SOLO si todo lo anterior salió bien ──
        # (si algo falló antes, el "with" registra FAILED y no se llega acá,
        #  así la próxima ejecución vuelve a leer las mismas filas)
        (spark.createDataFrame(
                [(SOURCE, TARGET, nuevo_wm, run.run_id)],
                "source_table string, target_table string, watermark_ts timestamp, run_id string")
            .withColumn("updated_at", F.current_timestamp())
            .write.mode("append").saveAsTable(WATERMARKS))

print(f"Leídas: {run.rows_read} | escritas en silver: {run.rows_written} | rechazadas: {run.rows_rejected}")

In [0]:
%sql
-- ¿Una fila por venta y con las correcciones aplicadas?
--   duplicados          → debe ser 0
--   ventas_modificadas  → debe ser 20 (las correcciones del lote 3)
SELECT
  COUNT(*)                                   AS filas,
  COUNT(DISTINCT transaction_id)             AS ventas_distintas,
  COUNT(*) - COUNT(DISTINCT transaction_id)  AS duplicados,
  COUNT(updated_at)                          AS ventas_modificadas
FROM workspace.silver.transactions;

In [0]:
%sql
-- Registros rechazados por regla
SELECT rule_name, COUNT(*) AS casos
FROM workspace.ops.data_quality_issues
GROUP BY rule_name
ORDER BY casos DESC;